In [1]:
# DAY 3: FEATURE ENGINEERING & PREPROCESSING (IMPROVED)
# UCI Heart Disease Prediction
# PURPOSE:
#   - Split data before preprocessing
#   - Fit imputers/encoders/scalers on training data only
#   - Transform train and test data safely
#   - Save processed data for modeling

import pandas as pd
import numpy as np
import os
import json
import pickle
from datetime import datetime

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
# SECTION 1: LOAD RAW DATA

print("LOADING RAW DATA")

df = pd.read_csv('data/heart_disease_uci.csv', na_values='?')
print(f"✓ Loaded: {df.shape[0]} rows × {df.shape[1]} columns")


LOADING RAW DATA
✓ Loaded: 920 rows × 16 columns


In [3]:
# SECTION 2: SEPARATE FEATURES AND TARGET

print("SEPARATING FEATURES AND TARGET")

# Define target
target_col = 'num'
y = df[target_col].copy()

# Define features (exclude ID and target)
X = df.drop(columns=['id', target_col])

print(f"✓ Features: {X.shape}")
print(f"✓ Target: {y.shape}")


SEPARATING FEATURES AND TARGET
✓ Features: (920, 14)
✓ Target: (920,)


In [4]:
# SECTION 3: IDENTIFY FEATURE TYPES

print("FEATURE TYPE IDENTIFICATION")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.columns[~X.columns.isin(numeric_features)].tolist()

print(f"\nNumeric features ({len(numeric_features)}):")
for col in numeric_features:
    print(f"  • {col}")

print(f"\nCategorical features ({len(categorical_features)}):")
for col in categorical_features:
    print(f"  • {col}")


FEATURE TYPE IDENTIFICATION

Numeric features (6):
  • age
  • trestbps
  • chol
  • thalch
  • oldpeak
  • ca

Categorical features (8):
  • sex
  • dataset
  • cp
  • fbs
  • restecg
  • exang
  • slope
  • thal


In [5]:
# SECTION 4: TRAIN-TEST SPLIT BEFORE PREPROCESSING

print("SPLITTING DATA BEFORE PREPROCESSING")

# IMPORTANT:
# Split first so imputation, encoding, and scaling learn only from the training set.
# This prevents data leakage from the test set into the training process.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"✓ Training features: {X_train_raw.shape}")
print(f"✓ Testing features:  {X_test_raw.shape}")
print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())


SPLITTING DATA BEFORE PREPROCESSING
✓ Training features: (736, 14)
✓ Testing features:  (184, 14)

Training target distribution:
num
0    329
1    212
2     87
3     86
4     22
Name: count, dtype: int64

Testing target distribution:
num
0    82
1    53
2    22
3    21
4     6
Name: count, dtype: int64


In [6]:
# SECTION 5: BUILD PREPROCESSING PIPELINE

print("BUILDING PREPROCESSING PIPELINE")

# Numeric pipeline: fill missing values, then z-normalize using training statistics only.
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: fill missing values, then one-hot encode nominal categories.
# handle_unknown='ignore' keeps the test transform safe if a new category appears.
try:
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', one_hot_encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

print(f"✓ Numeric preprocessing: median imputation + StandardScaler for {len(numeric_features)} features")
print(f"✓ Categorical preprocessing: most-frequent imputation + OneHotEncoder for {len(categorical_features)} features")


BUILDING PREPROCESSING PIPELINE
✓ Numeric preprocessing: median imputation + StandardScaler for 6 features
✓ Categorical preprocessing: most-frequent imputation + OneHotEncoder for 8 features


In [7]:
# SECTION 6: FIT ON TRAINING DATA AND TRANSFORM BOTH SPLITS

print("FITTING PREPROCESSOR ON TRAINING DATA ONLY")

X_train_processed_array = preprocessor.fit_transform(X_train_raw)
X_test_processed_array = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train_raw.index
).reset_index(drop=True)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=feature_names,
    index=X_test_raw.index
).reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Combined processed data is saved only as a convenience/inspection artifact.
# Model training should use the explicit train/test files saved below.
X_processed = pd.concat([X_train_processed, X_test_processed], axis=0, ignore_index=True)
y_processed = pd.concat([y_train, y_test], axis=0, ignore_index=True)

print(f"✓ Processed training data: {X_train_processed.shape}")
print(f"✓ Processed testing data:  {X_test_processed.shape}")
print("\nSample processed values (first 5 training rows):")
print(X_train_processed.head())


FITTING PREPROCESSOR ON TRAINING DATA ONLY
✓ Processed training data: (736, 29)
✓ Processed testing data:  (184, 29)

Sample processed values (first 5 training rows):
        age  trestbps      chol    thalch   oldpeak        ca  sex_Female  \
0 -0.033457 -0.627408 -1.799727 -1.700405 -0.803096 -0.361371         0.0   
1  2.195486 -0.091274 -1.799727  0.084842 -0.329703 -0.361371         0.0   
2 -0.033457 -0.091274 -1.799727 -0.113518  0.143690 -0.361371         0.0   
3  0.921804 -0.895475 -1.799727 -2.612865 -1.276489 -0.361371         0.0   
4  0.921804  1.409901 -0.248354  0.005498 -0.803096 -0.361371         0.0   

   sex_Male  dataset_Cleveland  dataset_Hungary  ...  restecg_normal  \
0       1.0                0.0              0.0  ...             1.0   
1       1.0                0.0              0.0  ...             1.0   
2       1.0                0.0              0.0  ...             0.0   
3       1.0                0.0              0.0  ...             1.0   
4       1.

In [8]:
# SECTION 7: SAVE PROCESSED DATA

print("SAVING PROCESSED DATA")

os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)

# Save leakage-safe train/test splits for modeling and evaluation
X_train_processed.to_csv('data/processed/X_train_processed.csv', index=False)
X_test_processed.to_csv('data/processed/X_test_processed.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

# Save combined processed files for quick inspection/backward compatibility
X_processed.to_csv('data/processed/X_processed.csv', index=False)
y_processed.to_csv('data/processed/y_processed.csv', index=False)

# Save fitted preprocessor so future raw rows can be transformed the same way
with open('outputs/models/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)

print("✓ Saved: data/processed/X_train_processed.csv")
print("✓ Saved: data/processed/X_test_processed.csv")
print("✓ Saved: data/processed/y_train.csv")
print("✓ Saved: data/processed/y_test.csv")
print("✓ Saved: outputs/models/preprocessor.pkl")

print(f"\nProcessed data shapes:")
print(f"  Train features: {X_train_processed.shape}")
print(f"  Test features:  {X_test_processed.shape}")
print(f"  Train target:   {y_train.shape}")
print(f"  Test target:    {y_test.shape}")


SAVING PROCESSED DATA
✓ Saved: data/processed/X_train_processed.csv
✓ Saved: data/processed/X_test_processed.csv
✓ Saved: data/processed/y_train.csv
✓ Saved: data/processed/y_test.csv
✓ Saved: outputs/models/preprocessor.pkl

Processed data shapes:
  Train features: (736, 29)
  Test features:  (184, 29)
  Train target:   (736,)
  Test target:    (184,)


In [9]:
# SECTION 8: SAVE PREPROCESSING METADATA

print("SAVING PREPROCESSING METADATA")

os.makedirs('outputs/reports', exist_ok=True)

preprocessing_metadata = {
    'preprocessing_date': datetime.now().isoformat(),
    'original_shape': {
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1])
    },
    'processed_shape': {
        'train_features': list(X_train_processed.shape),
        'test_features': list(X_test_processed.shape),
        'train_target': list(y_train.shape),
        'test_target': list(y_test.shape)
    },
    'feature_types': {
        'numeric': numeric_features,
        'categorical': categorical_features,
        'processed_feature_count': int(len(feature_names)),
        'processed_features': feature_names.tolist()
    },
    'preprocessing_steps': [
        'Train-test split before preprocessing',
        'Numeric missing value imputation fitted on training data only',
        'Categorical missing value imputation fitted on training data only',
        'Categorical one-hot encoding fitted on training data only',
        'Numeric scaling with StandardScaler fitted on training data only'
    ],
    'data_split': {
        'test_size': 0.2,
        'random_state': 42,
        'stratified_by': target_col
    },
    'target_distribution': {
        'train': {str(k): int(v) for k, v in y_train.value_counts().sort_index().items()},
        'test': {str(k): int(v) for k, v in y_test.value_counts().sort_index().items()}
    }
}

with open('outputs/reports/day3_preprocessing_metadata.json', 'w') as f:
    json.dump(preprocessing_metadata, f, indent=2)

print("✓ Saved: day3_preprocessing_metadata.json")


SAVING PREPROCESSING METADATA
✓ Saved: day3_preprocessing_metadata.json


In [10]:
# SECTION 9: DATA QUALITY VERIFICATION

print("DATA QUALITY VERIFICATION")

# Check no missing values remain
train_missing = int(X_train_processed.isnull().sum().sum())
test_missing = int(X_test_processed.isnull().sum().sum())
print(f"\nMissing values after preprocessing:")
print(f"  Train: {train_missing}")
print(f"  Test:  {test_missing}")

if train_missing == 0 and test_missing == 0:
    print("✓ No missing values!")
else:
    print("⚠ Missing values remain")

# Check data types
print(f"\nData types in processed training features:")
print(X_train_processed.dtypes.value_counts())

# Check target distribution
print(f"\nTraining target distribution:")
print(y_train.value_counts().sort_index())
print(f"\nTesting target distribution:")
print(y_test.value_counts().sort_index())


DATA QUALITY VERIFICATION

Missing values after preprocessing:
  Train: 0
  Test:  0
✓ No missing values!

Data types in processed training features:
float64    29
Name: count, dtype: int64

Training target distribution:
num
0    329
1    212
2     87
3     86
4     22
Name: count, dtype: int64

Testing target distribution:
num
0    82
1    53
2    22
3    21
4     6
Name: count, dtype: int64


In [11]:
# SECTION 10: SAVE PREPROCESSING CONFIGURATION

print("PREPROCESSING CONFIGURATION")

config = {
    'preprocessor_path': 'outputs/models/preprocessor.pkl',
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'processed_features': feature_names.tolist(),
    'target_column': target_col,
    'split': {
        'test_size': 0.2,
        'random_state': 42,
        'stratify': True
    },
    'rules': {
        'fit_preprocessing_on': 'training data only',
        'transform_test_with': 'the fitted training preprocessor',
        'numeric_imputation': 'median',
        'numeric_scaling': 'StandardScaler',
        'categorical_imputation': 'most_frequent',
        'categorical_encoding': 'OneHotEncoder(handle_unknown=ignore)'
    }
}

with open('outputs/reports/day3_preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Saved: day3_preprocessing_config.json")


PREPROCESSING CONFIGURATION
✓ Saved: day3_preprocessing_config.json


In [12]:
# COMPLETION

print("PREPROCESSING DONE")

print(f"""
Summary:
├─ ✓ Raw data shape        : {df.shape}
├─ ✓ Training features     : {X_train_processed.shape}
├─ ✓ Testing features      : {X_test_processed.shape}
├─ ✓ Numeric features      : {len(numeric_features)}
├─ ✓ Categorical features  : {len(categorical_features)}
├─ ✓ Missing values fixed  : Yes, using training-fitted imputers
├─ ✓ Categorical encoded   : Yes, using OneHotEncoder
├─ ✓ Numeric scaled        : Yes, after train-test split
└─ ✓ Data ready for ML     : Yes
""")


PREPROCESSING DONE

Summary:
├─ ✓ Raw data shape        : (920, 16)
├─ ✓ Training features     : (736, 29)
├─ ✓ Testing features      : (184, 29)
├─ ✓ Numeric features      : 6
├─ ✓ Categorical features  : 8
├─ ✓ Missing values fixed  : Yes, using training-fitted imputers
├─ ✓ Categorical encoded   : Yes, using OneHotEncoder
├─ ✓ Numeric scaled        : Yes, after train-test split
└─ ✓ Data ready for ML     : Yes

